# Advanced Usage Tutorial: Manual Type Selection + Pre-Propagation

This notebook demonstrates an advanced workflow on an **atomic** dataset:

1. Load an atomic HIN (e.g. `dhgl.get_dataset('atomic-imdb')`).
2. Manually assign node features with different formats (e.g. dense, sparse_coo, or even random embeddings).
3. Run `dhgl.prepropagate` to produce propagated (and concatenated) features.
4. Remove selected edge types with `transforms.remove_etypes`, while handling unreachable nodes.

In [1]:
import torch

import dhgl
from dhgl import transforms
from dhgl.type import NType

/home/smark/miniconda3/envs/graph/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Configure type selections

- `ntype_selections` controls how to **construct features** for each node type.
  - `'none'`: leave unchanged (no manual feature assignment).
  - `'nid'`: use a dense identity matrix as features.
  - `'nid_coo'`: use a sparse COO identity matrix as features.

- `etype_selections` controls which edge types to **keep** (`True`) vs **drop** (`False`).


In [2]:
ntype_selections = {
    'keyword': 'none',
    'director': 'nid_coo',
    'actor': 'none',
    'color': 'none',
    'language': 'nid',
    'country': 'nid',
    'content_rating': 'nid',
    'numerical': 'nid',
    'word': 'nid_coo',
    'movie': 'none',
}

etype_selections = {
    'is-type-of': False,
    'has-color': False,
    'is-country-of': False,
    'is-from-country': False,
    'is-language-of': False,
    'is-in-language': False,
}


## 2) Load the atomic dataset

In [11]:
hg = dhgl.get_dataset('atomic-imdb')
hg

Graph(num_nodes={'actor': 6124, 'color': 3, 'content_rating': 16, 'country': 65, 'director': 2393, 'keyword': 7971, 'language': 48, 'movie': 4932, 'numerical': 16, 'word': 3341},
      num_edges={('actor', 'acts', 'movie'): 14779, ('color', 'is-type-of', 'movie'): 4932, ('content_rating', 'is-rating-for', 'movie'): 4932, ('country', 'is-country-of', 'movie'): 4932, ('director', 'directed', 'movie'): 4932, ('keyword', 'is-in', 'movie'): 23610, ('language', 'is-language-of', 'movie'): 4932, ('movie', 'contains', 'keyword'): 23610, ('movie', 'contains-word', 'word'): 31335, ('movie', 'directed-by', 'director'): 4932, ('movie', 'has-color', 'color'): 4932, ('movie', 'has-numerical', 'numerical'): 78912, ('movie', 'has-rating', 'content_rating'): 4932, ('movie', 'is-from-country', 'country'): 4932, ('movie', 'is-in-language', 'language'): 4932, ('movie', 'stars', 'actor'): 14779, ('numerical', 'is-numerical-of', 'movie'): 78912, ('word', 'is-word-of', 'movie'): 31335},
      metagraph=[('ac

## 3) Assign node features (dense vs. sparse)

This block assigns per-type node features to `hg.nodes[ntype].data['feat']`.
The identity feature choice is useful for illustrating different storage formats.


In [12]:
for ntype, feat_type in ntype_selections.items():
    if feat_type == 'none':
        continue

    if feat_type == 'nid':
        # Dense identity features
        hg.nodes[ntype].data['feat'] = torch.eye(hg.num_nodes(ntype))

    elif feat_type == 'nid_coo':
        # Sparse COO identity features
        idx = torch.arange(hg.num_nodes(ntype))
        hg.nodes[ntype].data['feat'] = torch.sparse_coo_tensor(
            torch.stack([idx, idx]),
            torch.ones(hg.num_nodes(ntype)),
        )

    else:
        raise NotImplementedError(f'Unknown feature type: {feat_type}')

# Quick sanity check: list node types with assigned features
[(ntype, 'feat' in hg.nodes[ntype].data) for ntype in hg.ntypes]

[('actor', False),
 ('color', False),
 ('content_rating', True),
 ('country', True),
 ('director', True),
 ('keyword', False),
 ('language', True),
 ('movie', False),
 ('numerical', True),
 ('word', True)]

## 4) Pre-propagation

`dhgl.prepropagate` propagates features over the graph using edge weights.
By default, it returns **concatenated** features in `hg.ndata['feat']`.

If you prefer the pre-concatenation representation, you can pass `return_slots=True`.


In [14]:
feats: dict[NType, dict[NType, torch.Tensor]] = dhgl.prepropagate(hg, hg.ndata['feat'], hg.edata['weight'], return_slots=True)

{dsttype: {srctype: feat.shape for srctype, feat in nfeat.items()} for dsttype, nfeat in feats.items()}

{'content_rating': {'content_rating': torch.Size([16, 16]),
  'country': torch.Size([16, 65]),
  'director': torch.Size([16, 2393]),
  'language': torch.Size([16, 48]),
  'numerical': torch.Size([16, 16]),
  'word': torch.Size([16, 3341])},
 'movie': {'content_rating': torch.Size([4932, 16]),
  'country': torch.Size([4932, 65]),
  'director': torch.Size([4932, 2393]),
  'language': torch.Size([4932, 48]),
  'numerical': torch.Size([4932, 16]),
  'word': torch.Size([4932, 3341])},
 'actor': {'content_rating': torch.Size([6124, 16]),
  'country': torch.Size([6124, 65]),
  'director': torch.Size([6124, 2393]),
  'language': torch.Size([6124, 48]),
  'numerical': torch.Size([6124, 16]),
  'word': torch.Size([6124, 3341])},
 'color': {'content_rating': torch.Size([3, 16]),
  'country': torch.Size([3, 65]),
  'director': torch.Size([3, 2393]),
  'language': torch.Size([3, 48]),
  'numerical': torch.Size([3, 16]),
  'word': torch.Size([3, 3341])},
 'country': {'content_rating': torch.Size([65

By default, it returns **concatenated** features.

In [ ]:
# Default: return concatenated features
hg.ndata['feat'] = dhgl.prepropagate(hg, hg.ndata['feat'], hg.edata['weight'])

[(ntype, hg.nodes[ntype].data['feat'].shape) for ntype in hg.ntypes]

/home/smark/dhgl/src/dhgl/schema/preprop.py:546: UserWarning: Detected both sparse and dense features during prepropagation. This results in the creation of a custom torch.Tensor extension: MixedTensor. MixedTensor is experimental and currently only supports being passed to torch.nn.Linear. Use with caution.
  warnings.warn(


[('actor', torch.Size([6124, 5879])),
 ('color', torch.Size([3, 5879])),
 ('content_rating', torch.Size([16, 5879])),
 ('country', torch.Size([65, 5879])),
 ('director', torch.Size([2393, 5879])),
 ('keyword', torch.Size([7971, 5879])),
 ('language', torch.Size([48, 5879])),
 ('movie', torch.Size([4932, 5879])),
 ('numerical', torch.Size([16, 5879])),
 ('word', torch.Size([3341, 5879]))]

### MixedTensor (mixed dense/sparse feature formats)

If the propagated features contain **multiple storage formats** (e.g., some node types use dense tensors while others use sparse COO tensors), `dhgl.prepropagate(...)` will return a **`MixedTensor`** for the auto-concatenated output.

`MixedTensor` is designed to be *drop-in* for the common case where features are immediately passed through a **linear layer**. It currently supports only:

* `torch.nn.Linear`
* `torch.nn.functional.linear`

This means you typically **do not need to manually unify formats**, since most models apply a linear projection first (after which the output becomes a standard dense tensor).

Conceptually, the linear projection works as:

$$
\mathrm{linear}(X_{\text{dense}} \parallel X_{\text{sparse}})
= (X_{\text{dense}} \parallel X_{\text{sparse}}) W + b
= X_{\text{dense}} W_{\text{dense}} + X_{\text{sparse}} W_{\text{sparse}} + b,
$$
where $W_{\text{dense}} = W[:\mathrm{len}(X_{\text{dense}})]$ and $W_{\text{sparse}} = W[\mathrm{len}(X_{\text{dense}}):]$.

If you need operations beyond this supported interface, it is recommended to use `return_slots=True` and manually handle the propagated features (instead of relying on the default auto-concatenation).


## 5) Drop edge types while handling unreachable nodes

We remove edge types marked as `False` in `etype_selections`.

`unreachable_mode='nfeat'` keeps unreachable nodes **but drops their node features**.


In [15]:
hg = transforms.remove_etypes(
    hg,
    [etype for etype, selected in etype_selections.items() if not selected],
    unreachable_mode='nfeat',
)
hg

Graph(num_nodes={'actor': 6124, 'color': 3, 'content_rating': 16, 'country': 65, 'director': 2393, 'keyword': 7971, 'language': 48, 'movie': 4932, 'numerical': 16, 'word': 3341},
      num_edges={('actor', 'acts', 'movie'): 14779, ('content_rating', 'is-rating-for', 'movie'): 4932, ('director', 'directed', 'movie'): 4932, ('keyword', 'is-in', 'movie'): 23610, ('movie', 'contains', 'keyword'): 23610, ('movie', 'contains-word', 'word'): 31335, ('movie', 'directed-by', 'director'): 4932, ('movie', 'has-numerical', 'numerical'): 78912, ('movie', 'has-rating', 'content_rating'): 4932, ('movie', 'stars', 'actor'): 14779, ('numerical', 'is-numerical-of', 'movie'): 78912, ('word', 'is-word-of', 'movie'): 31335},
      metagraph=[('actor', 'movie', 'acts'), ('movie', 'keyword', 'contains'), ('movie', 'word', 'contains-word'), ('movie', 'director', 'directed-by'), ('movie', 'numerical', 'has-numerical'), ('movie', 'content_rating', 'has-rating'), ('movie', 'actor', 'stars'), ('content_rating', '

## 6) Inspect the final graph

In [16]:
print(dhgl.info(hg))


Dimensions of node features:
ntype             #samples    feat_dim
--------------  ----------  ----------
actor                 6124           0
color                    3           0
content_rating          16          16
country                 65           0
director              2393        2393
keyword               7971           0
language                48           0
movie                 4932           0
numerical               16          16
word                  3341        3341
Edges:
src_ntype       etype            dst_ntype         #edges    out_deg    in_deg
--------------  ---------------  --------------  --------  ---------  --------
actor           acts             movie              14779       2.41      3
content_rating  is-rating-for    movie               4932     308.25      1
director        directed         movie               4932       2.06      1
keyword         is-in            movie              23610       2.96      4.79
movie           contains      